# 01 - Exploratory Data Analysis
## Predicting Human Intestinal Absorption (HIA)
Mohammad Karamath Fardeen | 25251265 | MSc Data Science & Analytics, Maynooth University

This notebook explores the HIA_Hou dataset: class balance, distribution of molecular descriptors, and correlations between features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

SEED = 42

## 1. Load raw dataset splits

In [ ]:
train = pd.read_csv("../data/raw/hia_train.csv")
valid = pd.read_csv("../data/raw/hia_valid.csv")
test  = pd.read_csv("../data/raw/hia_test.csv")

print(f"Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}")
print(f"Total: {len(train) + len(valid) + len(test)}")
train.head()

## 2. Class distribution (HIA labels)

Y = 1 -> High intestinal absorption  
Y = 0 -> Low intestinal absorption

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, df) in zip(axes, [("Train", train), ("Valid", valid), ("Test", test)]):
    counts = df["Y"].value_counts().sort_index()
    bars = ax.bar(["Low (0)", "High (1)"], counts.values, color=["#E74C3C", "#2ECC71"])
    ax.set_title(f"{name} set (n={len(df)})")
    ax.set_ylabel("Count")
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(count), ha='center', fontweight='bold')

plt.suptitle("HIA Class Distribution Across Splits", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../results/figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nClass imbalance ratio (train):")
print(train["Y"].value_counts(normalize=True).round(3) * 100, "%")

**Observation:** The training set is heavily imbalanced (~89% high absorption vs. ~11% low absorption), motivating the use of SMOTE during model training.

## 3. Load processed features (RDKit descriptors)

In [ ]:
train_feat = pd.read_csv("../data/processed/hia_train_features.csv")

FEATURE_COLS = [
    "MolWt", "LogP", "TPSA", "HBD", "HBA",
    "RotBonds", "RingCount", "AromaticRings",
    "HeavyAtoms", "FractionCSP3", "MolMR", "NumHeteroatoms"
]

train_feat[FEATURE_COLS + ["Y"]].describe()

## 4. Feature distributions by absorption class

Comparing the distribution of key descriptors between high- and low-absorption molecules.

In [ ]:
key_features = ["TPSA", "LogP", "MolWt", "HBD", "HBA", "RotBonds"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    sns.kdeplot(data=train_feat, x=feat, hue="Y", fill=True, alpha=0.4, ax=ax,
                palette={0: "#E74C3C", 1: "#2ECC71"})
    ax.set_title(feat)
    ax.legend(title="HIA", labels=["High (1)", "Low (0)"])

plt.suptitle("Distribution of Key Molecular Descriptors by HIA Class", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("../results/figures/feature_distributions_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** TPSA shows the clearest separation between classes — low-absorption molecules are concentrated at higher TPSA values, consistent with known pharmacokinetic theory and later confirmed by SHAP analysis.

## 5. Correlation heatmap between descriptors

In [ ]:
plt.figure(figsize=(10, 8))
corr = train_feat[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
            cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix — Molecular Descriptors", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../results/figures/feature_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** As expected, Molecular Weight correlates strongly with Heavy Atom Count and Molar Refractivity (size-related descriptors). HBA and TPSA also show moderate positive correlation, since both relate to polar surface chemistry. No pair is so highly correlated (>0.95) as to require feature removal before modelling.

## 6. Boxplots: feature spread by class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    sns.boxplot(data=train_feat, x="Y", y=feat, ax=ax, palette={0: "#E74C3C", 1: "#2ECC71"})
    ax.set_xticklabels(["Low (0)", "High (1)"])
    ax.set_title(feat)

plt.suptitle("Boxplots of Key Descriptors by HIA Class", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("../results/figures/feature_boxplots_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary of EDA Findings

1. The dataset is imbalanced (~89% high absorption), requiring SMOTE or class weighting during model training.
2. TPSA shows the clearest visual separation between high- and low-absorption molecules, foreshadowing its dominance in the later SHAP feature importance analysis.
3. H-Bond Donors and Acceptors also show visible separation, consistent with Lipinski's Rule of Five.
4. No problematic multicollinearity was found among the 12 RDKit descriptors, so all were retained for modelling.